# Libraries Import

In [1]:
import os
import boto3
import requests
from pathlib import Path
from botocore.exceptions import NoCredentialsError, ClientError

# CPM Connection and Extraction

In [2]:
# These are CPM credentials that were created inside the application
url = 'https://envalior.symbioweb.com/envalior/Processworld/_api/rest/facets/processes/views/list/elements'

headers = {
    "symbio-auth-token": "REDACTED_SYMBIO_TOKEN",
    "api-version": "2.0",
    "Accept": "application/json"
}

TOKEN = "REDACTED_SYMBIO_TOKEN"

# Now we extract the name of all processes 
response = requests.get(url, headers=headers)
response.raise_for_status()

views = response.json()

In [3]:
views['count']

650

# Back-up

In [ ]:
# Now we create the code that extract the processes
def export_bpmn(process_id, name, token):
    url = (
        f"https://envalior.symbioweb.com/"
        f"envalior/Processworld/_api/v1/bpmn/{process_id}"
        f"?lcid=0&symbio-auth-token={token}"
    )

    headers = {
        "Accept": "application/xml"
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    Path('Saved processes').mkdir(exist_ok=True)

    filename = f"Saved processes/{name}.bpmn"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(response.text)

    print(f"Saved BPMN: {filename} into: Saved processes")

In [ ]:
# the files are deleted from the local folder
for root, dirs, files in os.walk(folder):
    for file in files:
        if file.endswith(".bpmn"):
            file_path = os.path.join(root, file)
            
            try:
                os.remove(file_path)
                print(f"Deleted: {file_path}")
            except Exception as e:
                print(f"Error deleting {file_path}: {e}")

In [ ]:
# Here we do the loop. we only take into consideration the processes that are in process
# since these are the ones that have the risk to be deleted. 
for element in views['elements']:
    if element.get('state') == 'inProcess' and element.get('type') == 'subProcess':
        versionId = element['versionId']
        name = list(element['attributes']['name'].values())[0]
        name = name.replace("/", " - ")
        try:
            export_bpmn(versionId, name, TOKEN)
        except ValueError:
            print(f'error with {name}')

# AWS Connection and data Push

In [ ]:
s3 = boto3.client(
    "s3",
    aws_access_key_id= "REDACTED_AWS_ACCESS_KEY",
    aws_secret_access_key= "REDACTED_AWS_SECRET_KEY",
    region_name='eu-west-1'
)

folder = "CPM/Saved processes"
bucket = "supplychain-mdm-prd-355747571374-celonis-cpm"

In [ ]:
# we first delete the files. since the back-up is expected to run periodically, what we want is to have only the latest version
# so what we do is first delete the last version and the upload the new one
response = s3.list_objects_v2(Bucket=bucket)

if "Contents" in response:
    objects_to_delete = [{"Key": obj["Key"]} for obj in response["Contents"]]

    s3.delete_objects(
        Bucket=bucket,
        Delete={"Objects": objects_to_delete}
    )

    print("All files deleted from bucket")

else:
    print("Bucket is already empty")

In [ ]:
for root, dirs, files in os.walk('Saved processes'):
    for file in files:
        if file.endswith(".bpmn"):
            local_path = os.path.join(root, file)
            try:
                s3.upload_file(local_path, bucket, file)
                print(f"Uploaded: {file} -> s3://{bucket}/{file}")
            except FileNotFoundError:
                print(f"Not found files: {local_path}")
            except NoCredentialsError:
                print("Credentials error")
            except ClientError as e:
                print(f"AWS Error: {e}")

In [ ]:
objects = s3.list_objects_v2(
    Bucket=bucket
)["Contents"]
objects